In [11]:
import os, sys, shutil
!pip install --upgrade uv

# 1. Clear the environment completely
!pip uninstall -y unsloth unsloth_zoo vllm trl transformers bitsandbytes

# 2. Physically wipe the site-packages of the troublemakers
# This is more effective than 'pip uninstall' which can leave ghost files
for path in sys.path:
    if 'dist-packages' in path:
        for pkg in ['unsloth', 'unsloth_zoo', 'trl', 'vllm', 'transformers', 'bitsandbytes']:
            target = os.path.join(path, pkg)
            if os.path.exists(target): shutil.rmtree(target)

# 3. Install dependencies that don't conflict first
#!uv pip install \
#    "torch>=2.11.0" \
#    "torchcodec==0.11.1" \
#    "torchvision>=0.26.0" \
#    "bitsandbytes" \
#    "transformers>=5.5.0" \
#    "tokenizers>=0.22.0"

# 3a. Create a constraints file to force versions
with open("overrides.txt", "w") as f:
    f.write("torchvision==0.26.0\n")
    f.write("torchaudio==2.11.0\n")
    f.write("torch==2.11.0\n")
    f.write("transformers>=5.5.0")
#    f.write("vllm==0.16.0\n")
    
# 3b. Install conflicting dependencies "Golden Stack"
#!uv pip install --system --upgrade --force-reinstall \
!uv pip install --system --overrides overrides.txt \
    "torch>=2.11.0" \
    "torchvision>=0.26.0" \
    "torchaudio>=2.11.0" \
    "torchcodec==0.11.1" \
    "bitsandbytes>=0.45.0" \
    "transformers>=5.5.0" \
    "trl>=0.28.0" \
    "vllm==0.16.0"

# 4. Force-install Unsloth Zoo WITHOUT its dependencies
# This prevents it from pulling in TRL 0.24.0
!uv pip install --system --no-deps "unsloth-zoo[base] @ git+https://github.com/unslothai/unsloth-zoo"
# 3. Finalize Unsloth Core
!uv pip install --system --no-deps "unsloth[base] @ git+https://github.com/unslothai/unsloth"

# 3. Force-install TRL and vLLM ignoring the 'zoo' constraints
#!uv pip install "trl>=0.28.0" "vllm==0.16.0"

# 4. Install Unsloth WITHOUT checking dependencies 
# This prevents the "Unsatisfiable" error from stopping the install
#!uv pip install --no-deps --no-cache-dir \
#    "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
#    "unsloth[base] @ git+https://github.com/unslothai/unsloth"


Found existing installation: unsloth 2026.4.8
Uninstalling unsloth-2026.4.8:
  Successfully uninstalled unsloth-2026.4.8
Found existing installation: unsloth_zoo 2026.4.9
Uninstalling unsloth_zoo-2026.4.9:
  Successfully uninstalled unsloth_zoo-2026.4.9
Using Python 3.12.13 environment at: /usr
Resolved 174 packages in 322ms                                       
Prepared 1 package in 11ms                                               
Uninstalled 4 packages in 24ms
Installed 8 packages in 102ms                               
 + bitsandbytes==0.49.2
 - cuda-python==12.9.4
 + cuda-python==13.2.0
 - fsspec==2026.4.0
 + fsspec==2026.2.0
 - numpy==2.4.4
 + numpy==2.2.6
 - setuptools==81.0.0
 + setuptools==80.10.2
 + transformers==5.7.0
 + trl==1.3.0
 + vllm==0.16.0
Using Python 3.12.13 environment at: /usr
Resolved 1 package in 336ms                                          
Installed 1 package in 2ms.9 (from git+https://github.com/un
 + unsloth-zoo==2026.4.9 (from git+https://github.com/u

In [ ]:
import os, importlib.util
!pip install --upgrade uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; _numpy = f"numpy=={numpy.__version__}"; _pil = f"pillow=={PIL.__version__}"
    except: _numpy = "numpy"; _pil = "pillow"
    !uv pip install \
        "torch>=2.9.1" "torchcodec==0.9.0" {_numpy} {_pil} torchvision bitsandbytes \
        --no-binary unsloth,unsloth_zoo \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        "vllm==0.16.0"
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install unsloth
# Gemma 4 requires transformers >= 5.5.0
!uv pip install --upgrade --no-deps --no-binary unsloth,unsloth_zoo "transformers>=5.5.0" "tokenizers>=0.22.0,<=0.23.0" "trl>=0.28.0" "huggingface-hub>=1.5.0,<2.0" unsloth unsloth_zoo

In [1]:
import os, sys

# 1. Create a brand new directory that isn't inside vLLM
bridge_path = "/content/unsloth_vllm_bridge"
os.makedirs(bridge_path, exist_ok=True)
with open(os.path.join(bridge_path, "__init__.py"), "w") as f: f.write("")

# 2. Create the file Unsloth wants, but in our safe directory
with open(os.path.join(bridge_path, "openenv_utils.py"), "w") as f:
    f.write("def reload_weights(model, weights_path):\n    \"\"\"Source for Unsloth.\"\"\"\n    pass\n")

# 3. Hijack the system mapping
# We tell Python: "When someone asks for vllm.utils.openenv_utils, give them MY file."
import unsloth_vllm_bridge.openenv_utils
sys.modules["vllm.utils.openenv_utils"] = sys.modules["unsloth_vllm_bridge.openenv_utils"]

print("✅ System Hijack Complete. vllm.utils.openenv_utils now points to a readable .py file.")

✅ System Hijack Complete. vllm.utils.openenv_utils now points to a readable .py file.


In [2]:
import os

# 1. Memory Sharing: Makes vLLM and the Trainer share model weights
# Required to fit Gemma 4 + GRPO on 16GB-24GB GPUs.
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# 2. Patching Logic: Forces Unsloth to use the new 'OpenEnv' hooks 
# This is specifically what bridges the gap with vLLM 0.16.0+
os.environ["UNSLOTH_USE_OPENENV"] = "1"

# 3. Optimization: Prevents vLLM from pre-allocating 90% of VRAM
# letting Unsloth handle the dynamic memory balancing instead.
os.environ["VLLM_WORKER_MULTIPROC_METHOD"] = "spawn"

from unsloth import FastLanguageModel, PatchFastRL
import torch

# We manually inject the reference into the Unsloth global space 
# right before patching to make sure it doesn't try to look at the disk again.
import unsloth.models.rl_replacements as rl_repl
import vllm.utils.openenv_utils
rl_repl.openenv_utils = vllm.utils.openenv_utils

PatchFastRL("GRPO", FastLanguageModel)

import trl
print(f"TRL version: {trl.__version__}")
import torch
print(f"Torch version: {torch.__version__}")
import torchcodec
print("Torchcodec loaded successfully!")


🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


Please reinstall via `uv pip install unsloth vllm torchvision torchaudio --torch-backend=auto`.
ERROR:bitsandbytes.cextension:bitsandbytes library load error: libnvJitLink.so.13: cannot open shared object file: No such file or directory
Traceback (most recent call last):
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 320, in <module>
    lib = get_native_library()
          ^^^^^^^^^^^^^^^^^^^^
  File "/usr/local/lib/python3.12/dist-packages/bitsandbytes/cextension.py", line 298, in get_native_library
    dll = ct.cdll.LoadLibrary(str(binary_path))
          ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 460, in LoadLibrary
    return self._dlltype(name)
           ^^^^^^^^^^^^^^^^^^^
  File "/usr/lib/python3.12/ctypes/__init__.py", line 379, in __init__
    self._handle = _dlopen(self._name, mode)
                   ^^^^^^^^^^^^^^^^^^^^^^^^^
OSError: libnvJitLink.so.13: cannot open shared object file: No s

🦥 Unsloth Zoo will now patch everything to make training faster!


ModuleNotFoundError: No module named 'vllm'

In [ ]:
import vllm.utils
import inspect

print(f"1. vLLM Utils Location: {vllm.utils.__file__}")
# If this ends in .so, it's a binary block.

try:
    import vllm.utils.openenv_utils as ou
    print(f"2. OpenEnv Location: {ou.__file__}")
    source = inspect.getsource(ou.reload_weights)
    print("3. Source check: SUCCESS")
except Exception as e:
    print(f"3. Source check: FAILED - {e}")

In [ ]:
import os
import sys

# 1. Set variables BEFORE any imports
os.environ["UNSLOTH_USE_OPENENV"] = "1"
os.environ["UNSLOTH_VLLM_STANDBY"] = "1"

# 2. Re-apply our physical bridge (in case the restart cleared the sys.modules)
import vllm
import vllm.utils.openenv_utils
import unsloth.models.rl_replacements as rl_repl
rl_repl.openenv_utils = vllm.utils.openenv_utils

# 3. Now import and patch
from unsloth import FastLanguageModel, PatchFastRL
try:
    PatchFastRL("GRPO", FastLanguageModel)
    print("✅✅✅ FINAL SUCCESS! ✅✅✅")
except OSError:
    import inspect
    import trl
    print(f"TRL check: {inspect.getsourcefile(trl.GRPOTrainer)}")
    print("If the line above shows a path, try running THIS cell one more time.")

# Model Loading

In [ ]:
from unsloth import FastModel
import torch

max_seq_length = 1024 # Can increase for longer reasoning traces
lora_rank = 16 # Larger rank = smarter, but slower

model_name = "unsloth/gemma-4-E2B-it-unsloth-bnb-4bit"

model, tokenizer = FastModel.from_pretrained(
    model_name = model_name,
    dtype = None, # None for auto detection
    max_seq_length = max_seq_length, # Choose any for long context!
    load_in_4bit = True,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "YOUR_HF_TOKEN", # HF Token for gated models
)

In [ ]:
temperature = 0.8

from vllm import SamplingParams
import random
import io
from collections import Counter

sampling_params = SamplingParams(
    temperature = temperature,
    top_p = 0.95,
    max_tokens = 512,
)